# Tutorial 3.1: Simulation of FOV-Induced Incomplete Registration in Embryonic Mouse Brain

This tutorial uses paired spatial chromatin-accessibility and RNA profiles from embryonic mouse brain MISAR-seq data, available from the [original study](https://www.nature.com/articles/s41592-023-01884-1). In this source-to-target setting (ATAC-to-RNA), ATAC remains complete while a contiguous RNA region is masked to simulate FOV-induced incomplete registration.

The controlled mask represents technical loss of RNA coverage rather than biological RNA absence. Because the profiles are paired before masking, the original RNA values at target-unregistered locations provide location-matched ground truth for evaluating PRISM imputation.


In [ ]:
from pathlib import Path

import scanpy as sc
import PRISM
from PRISM import (plot_imputation_metric_boxplot, compute_similarity_prior, plot_prism_imputation_spatial,
                   preprocess_omics, prism_eval_and_save, run_clustering_eval_plot, select_best_device,
                   set_prism_plot_style, set_seed, simulate_missing_sliding)
set_prism_plot_style()

### Load data

The original dataset contains paired ATAC and RNA profiles from embryonic mouse brain at E13.5, E15.5 and E18.5. `Slice_ID` selects the stage used in this tutorial, with E15.5 as the default.


In [ ]:
DEVICE = select_best_device()
RANDOM_SEED = 2024
set_seed(RANDOM_SEED)
Slice_ID = "E15.5"

DATASET_DIR = Path("Datasets") / "embryonic mouse brain" / Slice_ID
SOURCE_H5AD = DATASET_DIR / "adata_atac.h5ad"
TARGET_H5AD = DATASET_DIR / "adata_rna.h5ad"
RESULTS_DIR = Path("Results") / "Tutorial3_1_embryonic_mouse_brain"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

PRIOR_PATH = RESULTS_DIR / f"{Slice_ID}_emb_mouse_AOT.npz"
RUN_PREFIX = f"{Slice_ID}_emb_mouse"

adata_source = sc.read_h5ad(SOURCE_H5AD)
adata_target = sc.read_h5ad(TARGET_H5AD)
adata_source.var_names_make_unique()
adata_target.var_names_make_unique()


### Simulating incomplete registration

`simulate_missing_sliding` provides a controlled simulation of source-to-target incomplete registration by withholding the target modality at selected locations while retaining the paired source modality. `missing='0'` denotes an RNA-unregistered location and `missing='1'` an observed ATAC-RNA pair; the original target values remain available only as ground truth for evaluation.

Here, a horizontal window covering 50% of locations is designated as RNA-unregistered while ATAC remains observed across the section. This models a technical RNA field-of-view mismatch, such as partial target-assay coverage or a shifted acquisition window, rather than biological RNA absence.


In [ ]:
missing_indices = simulate_missing_sliding(adata_target, spatial_key="spatial", direction="H",
                                           missing_width=0.50, step_ratio=0.10, step_id=0,
                                           label_key="missing", lock_at_end=True, point_size=4, 
                                           plot=True, figsize=(4, 4))

print(f"RNA missing cells: {len(missing_indices)}/{adata_target.n_obs}")

### Preprocessing source (ATAC) and target (RNA) modalities

`preprocess_omics` applies modality-specific preprocessing while retaining the availability label needed to distinguish observed and RNA-unregistered locations. ATAC is represented in latent semantic indexing (LSI) space for chromatin-neighbourhood modelling, whereas RNA is processed as the partially observed target.

In [ ]:
adata_source, _ = preprocess_omics(adata_source, modality="ATAC", missing_key="missing", n_peak=10000,
                                   n_comps=50, data_role="source", use_lsi_as_X=True)

adata_target, _ = preprocess_omics(adata_target, modality="RNA", missing_key="missing", min_cells=10,
                                   hvgs=3000, data_role="target", compute_pca=False, save_log_layer=True,
                                   save_raw_eval=True)

print("ATAC model shape after preprocessing:", adata_source.shape)
print("ATAC LSI shape:", adata_source.obsm["X_lsi"].shape)
print("RNA HVG shape after preprocessing:", adata_target.shape)

### Constructing the ATAC-LSI similarity prior

`compute_similarity_prior` constructs a spatial-niche prior from the complete source modality. Here, COVET represents each ATAC location through its local chromatin environment in `X_lsi`, and AOT yields a source-derived similarity matrix that PRISM uses to retrieve availability-compatible ATAC-RNA references for RNA-unregistered queries.

The RNA availability mask identifies which ATAC-RNA pairs remain observed for PRISM, while the fully observed ATAC profiles provide the molecular context used to form the prior.


In [ ]:
distance_matrix, prior_metrics = compute_similarity_prior(adata_source, adata_target, PRIOR_PATH,
                                                          device=DEVICE, covet_k_spatial=6, covet_gene_num=None,
                                                          covet_use_layer=None, covet_use_obsm="X_lsi", 
                                                          spatial_key="spatial", missing_key="missing", 
                                                          store_neighbor_index=True)

In [ ]:
# Constructing spatial graphs
PRISM.Cal_Spatial_Net(adata_source, rad_cutoff=1.5)
PRISM.Stats_Spatial_Net(adata_source)
PRISM.Cal_Spatial_Net(adata_target, rad_cutoff=1.5)
PRISM.Stats_Spatial_Net(adata_target)

### Training PRISM

In this ATAC-to-RNA setting, `missing` marks unavailable RNA profiles, which are represented internally by learnable imputation tokens. The ATAC-derived COVET/AOT prior retrieves `k_top` observed references, and the Transformer organizes each query with these references into a sequence that propagates chromatin-informed context to RNA-unregistered locations.

`n_epochs` sets the maximum training budget, and `min_epochs` together with `patience` governs loss-based early stopping. `center_drop_rate` temporarily masks observed RNA centres during training, `noise` introduces controlled spatial-graph perturbation, and optional `interaction_pca` appends PCA-compressed ATAC-RNA interaction features to `PRISM_emb`. The embedding supports spatial-domain identification (Task 1), while the RNA decoder supports missing-omics imputation (Task 2).


In [ ]:
# Train PRISM and add interaction principal components
adata_source_out, adata_target_out = PRISM.train_PRISM(adata_source, adata_target, distance_matrix,
                                                       k_top=5, n_epochs=1000, lr=1e-3,
                                                       output_dir=str(RESULTS_DIR), file_prefix=RUN_PREFIX, 
                                                       device=DEVICE, patience=20, min_epochs=50, 
                                                       center_drop_rate=0.1, noise=0.1,
                                                       load_model_path=False, interaction_pca=True)

### Task 1: Spatial-domain identification

In [ ]:
adata_clustered, domain_metrics = run_clustering_eval_plot(adata_source_out, emb_key="PRISM_emb", 
                                                           label_key="Combined_Clusters_annotation", 
                                                           cluster_key="PRISM_mclust", n_clusters=12, s=35,
                                                           use_pca=True, align_labels=True, 
                                                           aligned_key="PRISM_mclust_domain",
                                                           dataset_name="emb_mouse_e15.5")

### Task 2: RNA imputation

In [ ]:
# Evaluate RNA imputation on the top 800 HVGs
imputation_results = prism_eval_and_save(truth_adata=adata_target_out, adata=adata_target_out, 
                                         save_path=str(RESULTS_DIR), first_name=RUN_PREFIX, 
                                         missing_indices=missing_indices, topk_features=800,
                                         topk_rank_by="var_order", topk_only=True, save_topk_summary=True,
                                         save_files=False)

_ = plot_imputation_metric_boxplot(imputation_results, feature_names=adata_target.var_names,
                                   feature_label="RNA gene", output_suffix="RNA", plot_type="raincloud")

In [ ]:
# Visualize representative gene imputation
RNA_FEATURE_TO_PLOT = "ENSMUSG00000090386"
fig, axes = plot_prism_imputation_spatial(imputation_results=imputation_results, split1_indices=missing_indices,
                                                 feature=RNA_FEATURE_TO_PLOT, show_missing_only=False, 
                                                 highlight_missing=False, figsize=(8, 3))

feature_idx = adata_target.var_names.get_loc(RNA_FEATURE_TO_PLOT)
feature_metrics = {metric: round(float(imputation_results["raw"]["per_protein"][metric][feature_idx]), 4)
                   for metric in ("PCC", "SPCC", "MSE")}
print(f"Representative gene {RNA_FEATURE_TO_PLOT}: {feature_metrics}")